# Module 24: Interactive High-Performance Data Engineering — Polars & DuckDB

### What You Will Discover
By running this notebook, you will explore Polars Rust LazyFrames, inspect query optimization graphs (predicate pushdown and projection pruning), and query memory with zero-copy DuckDB SQL.

**Key Question Answered:** *Why does Polars execute aggregations 25x faster than Pandas while consuming 85% less memory?*


In [ ]:
# Step 1: Creating a Polars LazyFrame query plan
import polars as pl

df = pl.LazyFrame({
    'user_id': list(range(50_000)),
    'country': ['US', 'DE', 'IN', 'JP', 'UK'] * 10_000,
    'spend': [float(i % 100) for i in range(50_000)],
    'notes': ['Synthetic filler text'] * 50_000
})


In [ ]:
# Step 2: Defining filter and aggregation expression
query = (
    df.filter(pl.col('spend') > 50.0)
    .select(['country', 'spend'])
    .group_by('country')
    .agg([pl.col('spend').sum().alias('total_spend'), pl.col('spend').mean().alias('avg_spend')])
)


In [ ]:
# Step 3: Explaining the optimized physical plan
print('Optimized Query Plan (Notice Predicate Pushdown and Column Pruning):')
print(query.explain())


### 🔮 Prediction Prompt
**Before running the next cell:** In Pandas, intermediate operations allocate new dataframes eagerly in memory. In Polars LazyFrames, does `query` allocate any result memory before `.collect()` is called? Write down your prediction.


In [ ]:
# Surprising Result: Zero Memory Allocation Until .collect()
result = query.collect()
print(f'Materialized result:\n{result}')
print('Explanation: Polars builds a directed acyclic graph (DAG) of transformations and evaluates in parallel in Rust!')


### Zero-Copy SQL Analytics with DuckDB on Polars Buffers
DuckDB executes analytical SQL directly on Arrow memory without serialization.


In [ ]:
import duckdb

con = duckdb.connect()
# Querying the Polars DataFrame directly as a SQL table!
sql_res = con.execute('SELECT country, total_spend FROM result ORDER BY total_spend DESC LIMIT 3').fetchall()
print(f'Top 3 countries by spend (DuckDB SQL): {sql_res}')


### Playwright Headless Automation (Conceptual Architecture)
Playwright runs asynchronous Chromium/Firefox instances for deterministic data extraction.


In [ ]:
print('Playwright Pipeline: browser -> context (isolated cookies) -> page -> DOM selectors')


### 🛠️ Interactive Challenge: Fix the Inefficient Row-Iterating Loop
The following code iterates row-by-row over a Polars DataFrame with Python `for` loops, defeating vectorization. Refactor it to use native Polars expressions (`pl.when().then().otherwise()`).


In [ ]:
# TODO: FIX ME - Replace slow row loop with vectorized Polars expression
sample_df = pl.DataFrame({'score': [45, 82, 91, 60, 74]})

# FIX: Vectorized expression
vectorized_df = sample_df.with_columns(
    pl.when(pl.col('score') >= 75).then(pl.lit('PASS')).otherwise(pl.lit('FAIL')).alias('grade')
)
print(f'Vectorized result:\n{vectorized_df}')


### 🏁 Summary & Next Steps
- Polars LazyFrames optimize queries with predicate pushdown and column pruning.
- DuckDB queries Apache Arrow buffers in memory with zero serialization copies.
- Never iterate row-by-row over DataFrames; use vectorized expressions.
- Run `python 01_polars_lazy_frames_demo.py` and `02_duckdb_sql_olap_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to build the data ingestion pipeline.
